# Additional baselines and ablations

1. Periodic retraining with different schedules, including retraining triggered by the
   PHT+KS drift detector.
2. Other online learners in place of ARF.
3. Decision trees, to check how far simple threshold rules go on this dataset.
4. ARF with and without its internal drift detectors.
5. Sensitivity of ARF to its main hyperparameters.

Everything uses the same training set, stream construction and seeds as the main results.
Sections 2 and 5 are the slowest (roughly an hour together on a laptop); lower `N_SEEDS`
or `GRID_SEEDS` for a quicker check.

In [1]:
import itertools
import os
import time

import numpy as np
import pandas as pd
from river import ensemble, forest, naive_bayes, neighbors, preprocessing, tree
from sklearn.tree import DecisionTreeClassifier, export_text

from utils import (FEATURES, load_dataset, make_training_set, make_oracle_extra,
                   make_phased_stream, make_arf, train_river_model, run_stream,
                   run_batch_retraining, recovery_time, pht_ks_alarms, merge_close_alarms,
                   OnlineMLP)

DATA_DIR = "data"
SEED = 42
N_SEEDS = 10
GRID_SEEDS = 5

train_df, test_df = load_dataset(DATA_DIR)
X_train, y_train = make_training_set(train_df, seed=SEED)
X_extra, y_extra = make_oracle_extra(train_df, seed=SEED)
streams = {seed: make_phased_stream(test_df, seed=seed) for seed in range(N_SEEDS)}


def summarise(preds, y, starts):
    return {"errors": int((preds != y).sum()),
            "accuracy": (preds == y).mean(),
            "recovery after EDFA": recovery_time(y, preds, starts[1]),
            "recovery after NLI": recovery_time(y, preds, starts[2])}


def mean_std_table(df, by):
    out = df.drop(columns="seed").groupby(by, sort=False).agg(["mean", "std"])
    return out.round(4)

## 1. Retraining schedules

- every 5000 samples, at the start of each phase (as in the paper)
- every 5000 samples, shifted by 2500 so that each retrain falls in the middle of a phase
- every 1000 samples
- whenever the PHT+KS detector raises an alarm on any of the four features. This trigger
  uses only the telemetry, not the labels. Alarms closer than 800 samples are merged.

The online ARF is included for comparison.

In [2]:
def detector_alarms(X):
    alarms = []
    for feature in FEATURES:
        _, confirmed = pht_ks_alarms(X[feature].to_numpy())
        alarms += confirmed
    return merge_close_alarms(alarms)


rows = []
for seed, (X_stream, y_stream, starts) in streams.items():
    n = len(y_stream)
    schedules = {
        "every 5000 (paper)": starts[1:],
        "every 5000, offset 2500": list(range(2500, n, 5000)),
        "every 1000": list(range(1000, n, 1000)),
        "on PHT+KS alarm": detector_alarms(X_stream),
    }
    for name, retrain_at in schedules.items():
        preds, _, fit_times = run_batch_retraining(X_train, y_train, X_stream, y_stream,
                                                   retrain_at, seed)
        rows.append({"model": name, "seed": seed, "retrains": len(fit_times),
                     "retraining time (s)": sum(fit_times),
                     **summarise(preds, y_stream, starts)})

    arf = train_river_model(make_arf(seed), X_train, y_train)
    preds, _ = run_stream(arf, X_stream, y_stream)
    rows.append({"model": "online ARF", "seed": seed, "retrains": 0,
                 "retraining time (s)": 0.0, **summarise(preds, y_stream, starts)})
    print(f"seed {seed} done")

schedule_results = pd.DataFrame(rows)
schedule_results.to_csv("results_retraining_schedules.csv", index=False)
mean_std_table(schedule_results, "model")

seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done
seed 5 done
seed 6 done
seed 7 done
seed 8 done
seed 9 done


retrains         retraining time (s)          errors  \
                            mean     std                mean     std    mean   
model                                                                          
every 5000 (paper)           4.0  0.0000              0.2711  0.0154  2943.5   
every 5000, offset 2500      5.0  0.0000              0.3449  0.0195  1475.9   
every 1000                  24.0  0.0000              1.6726  0.1010   611.3   
on PHT+KS alarm              6.1  1.6633              0.4121  0.1177   224.1   
online ARF                   0.0  0.0000              0.0000  0.0000   104.2   

                                 accuracy         recovery after EDFA  \
                             std     mean     std                mean   
model                                                                   
every 5000 (paper)       52.3116   0.8823  0.0021                 NaN   
every 5000, offset 2500  38.9542   0.9410  0.0016              2495.8   
every 1000               21.9294   0.9755  0.0009              1015.5   
on PHT+KS alarm           8.9623   0.9910  0.0004               403.6   
online ARF               12.6210   0.9958  0.0005               254.0   

                                  recovery after NLI            
                              std               mean       std  
model                                                           
every 5000 (paper)            NaN                NaN       NaN  
every 5000, offset 2500    5.4528             2492.7    5.7164  
every 1000                60.6745             1002.9   28.3802  
on PHT+KS alarm           44.0005              227.1   48.8841  
online ARF               235.9374              102.3  118.0754

## 2. Other online learners

All River models use their default settings and the same test-then-train protocol as the
online ARF. kNN keeps the last 500 samples and works on standardised features. The MLP is
a small scikit-learn network (two hidden layers of 32 and 16 units) updated with one sample
at a time.

In [3]:
def make_learner(name, seed):
    if name == "ARF":
        return train_river_model(make_arf(seed), X_train, y_train)
    if name == "Incremental MLP":
        return OnlineMLP(seed=seed).fit_initial(X_train, y_train)
    models = {
        "Hoeffding tree": lambda: tree.HoeffdingTreeClassifier(),
        "Hoeffding adaptive tree": lambda: tree.HoeffdingAdaptiveTreeClassifier(seed=seed),
        "Gaussian naive Bayes": lambda: naive_bayes.GaussianNB(),
        # kNN is distance based, so the features are standardised first. Without
        # this the BER values (around -600 to -30) dominate every distance.
        "kNN (window 500)": lambda: preprocessing.StandardScaler() | neighbors.KNNClassifier(
            n_neighbors=5, engine=neighbors.LazySearch(window_size=500)),
        "Leveraging bagging": lambda: ensemble.LeveragingBaggingClassifier(
            model=tree.HoeffdingTreeClassifier(), seed=seed),
        "Streaming random patches": lambda: ensemble.SRPClassifier(seed=seed),
    }
    return train_river_model(models[name](), X_train, y_train)


LEARNERS = ["ARF", "Hoeffding tree", "Hoeffding adaptive tree", "Gaussian naive Bayes",
            "kNN (window 500)", "Leveraging bagging", "Streaming random patches",
            "Incremental MLP"]

rows = []
for seed, (X_stream, y_stream, starts) in streams.items():
    for name in LEARNERS:
        tic = time.perf_counter()
        preds, _ = run_stream(make_learner(name, seed), X_stream, y_stream)
        rows.append({"model": name, "seed": seed, "run time (s)": time.perf_counter() - tic,
                     **summarise(preds, y_stream, starts)})
    print(f"seed {seed} done")

learner_results = pd.DataFrame(rows)
learner_results.to_csv("results_online_learners.csv", index=False)
mean_std_table(learner_results, "model")

seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done
seed 5 done
seed 6 done
seed 7 done
seed 8 done
seed 9 done


run time (s)          errors           accuracy  \
                                 mean     std    mean       std     mean   
model                                                                      
ARF                           23.2049  1.5732   104.2   12.6210   0.9958   
Hoeffding tree                 1.4022  0.1062   284.5   42.7038   0.9886   
Hoeffding adaptive tree        3.6785  0.3673   346.5   85.1212   0.9861   
Gaussian naive Bayes           4.3568  0.3968    38.6    6.0590   0.9985   
kNN (window 500)              40.4118  1.5037  1171.0   26.8287   0.9532   
Leveraging bagging            61.3352  4.8401   158.3   13.9208   0.9937   
Streaming random patches      78.7105  7.3207   525.1  335.7798   0.9790   
Incremental MLP               26.7879  1.4896   125.1   18.9294   0.9950   

                                 recovery after EDFA            \
                             std                mean       std   
model                                                            
ARF                       0.0005               254.0  235.9374   
Hoeffding tree            0.0017               585.1   71.7626   
Hoeffding adaptive tree   0.0034               668.9  273.2067   
Gaussian naive Bayes      0.0002                31.2   47.8837   
kNN (window 500)          0.0011              1318.5  500.5533   
Leveraging bagging        0.0006               429.1   76.4016   
Streaming random patches  0.0134               668.7  452.5061   
Incremental MLP           0.0008                89.9   94.1210   

                         recovery after NLI             
                                       mean        std  
model                                                   
ARF                                102.3000   118.0754  
Hoeffding tree                     484.4000   215.0267  
Hoeffding adaptive tree            462.9000   202.9135  
Gaussian naive Bayes                10.1000     5.1737  
kNN (window 500)                   957.4000  1356.8059  
Leveraging bagging                 350.3000   233.8452  
Streaming random patches           987.6667  1452.1470  
Incremental MLP                    144.3000    98.8355

## 3. Decision trees

A depth-3 tree trained on the same Normal and ECL data as the other models, and a depth-3
tree that also sees the oracle's EDFA and NLI samples. Neither is updated on the stream.

In [4]:
tree_two_classes = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, y_train)
tree_all_classes = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(
    pd.concat([X_train, X_extra]), np.concatenate([y_train, y_extra]))

print("Trained on Normal and ECL:")
print(export_text(tree_two_classes, feature_names=FEATURES))
print("Trained on all four classes:")
print(export_text(tree_all_classes, feature_names=FEATURES))

Trained on Normal and ECL:
|--- Laser current (mA) <= 40.63
|   |--- class: 0
|--- Laser current (mA) >  40.63
|   |--- class: 1

Trained on all four classes:
|--- Laser current (mA) <= 40.63
|   |--- LP power (dBm) <= -2.03
|   |   |--- class: 2
|   |--- LP power (dBm) >  -2.03
|   |   |--- LP power (dBm) <= -1.98
|   |   |   |--- class: 0
|   |   |--- LP power (dBm) >  -1.98
|   |   |   |--- class: 2
|--- Laser current (mA) >  40.63
|   |--- LP power (dBm) <= -0.03
|   |   |--- class: 1
|   |--- LP power (dBm) >  -0.03
|   |   |--- class: 3



In [5]:
rows = []
for seed, (X_stream, y_stream, starts) in streams.items():
    for name, model in [("Decision tree, Normal + ECL", tree_two_classes),
                        ("Decision tree, all classes", tree_all_classes)]:
        preds = model.predict(X_stream)
        rows.append({"model": name, "seed": seed, **summarise(preds, y_stream, starts)})

tree_results = pd.DataFrame(rows)
tree_results.to_csv("results_decision_trees.csv", index=False)
mean_std_table(tree_results, "model")

errors          accuracy          \
                               mean      std     mean     std   
model                                                           
Decision tree, Normal + ECL  9982.0  76.5622   0.6007  0.0031   
Decision tree, all classes     37.3   5.5588   0.9985  0.0002   

                            recovery after EDFA          recovery after NLI  \
                                           mean      std               mean   
model                                                                         
Decision tree, Normal + ECL                 NaN      NaN                NaN   
Decision tree, all classes                 23.6  50.5529                0.0   

                                  
                             std  
model                             
Decision tree, Normal + ECL  NaN  
Decision tree, all classes   0.0

## 4. ARF with and without its internal drift detectors

Each ARF tree has an ADWIN warning detector and an ADWIN drift detector on its own error.
A warning starts a background tree, and a drift replaces the tree with it. Here the
warnings and drifts are logged during the stream, and the run is repeated with both
detectors switched off, so that the trees can only grow.

In [6]:
def run_with_detector_log(model, X, y):
    """Online run that also records when ARF raised warnings and drifts."""
    preds, warnings_at, drifts_at = [], [], []
    n_warnings = model.n_warnings_detected()
    n_drifts = model.n_drifts_detected()
    for t, (x, label) in enumerate(zip(X.to_dict("records"), y)):
        preds.append(model.predict_one(x))
        model.learn_one(x, int(label))
        if model.n_warnings_detected() > n_warnings:
            warnings_at.append(t)
            n_warnings = model.n_warnings_detected()
        if model.n_drifts_detected() > n_drifts:
            drifts_at.append(t)
            n_drifts = model.n_drifts_detected()
    return np.array(preds), warnings_at, drifts_at


AFTER_ONSET = 1000   # how many samples after an onset count as "near" it

rows = []
for seed, (X_stream, y_stream, starts) in streams.items():
    for detectors in (True, False):
        arf = train_river_model(make_arf(seed, drift_detection=detectors), X_train, y_train)
        preds, warnings_at, drifts_at = run_with_detector_log(arf, X_stream, y_stream)
        near = [t for t in drifts_at if any(s <= t < s + AFTER_ONSET for s in starts[1:])]
        rows.append({"model": "ARF with detectors" if detectors else "ARF without detectors",
                     "seed": seed, "warnings": len(warnings_at), "drifts": len(drifts_at),
                     "drifts near an onset": len(near), **summarise(preds, y_stream, starts)})
    print(f"seed {seed} done")

detector_results = pd.DataFrame(rows)
detector_results.to_csv("results_arf_detectors.csv", index=False)
mean_std_table(detector_results, "model")

seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done
seed 5 done
seed 6 done
seed 7 done
seed 8 done
seed 9 done


warnings         drifts         drifts near an onset  \
                          mean     std   mean     std                 mean   
model                                                                        
ARF with detectors        71.8  7.5836   67.6  7.0742                 34.2   
ARF without detectors      0.0  0.0000    0.0  0.0000                  0.0   

                              errors          accuracy          \
                          std   mean      std     mean     std   
model                                                            
ARF with detectors     5.0947  104.2  12.6210   0.9958  0.0005   
ARF without detectors  0.0000  317.0  21.6282   0.9873  0.0009   

                      recovery after EDFA           recovery after NLI  \
                                     mean       std               mean   
model                                                                    
ARF with detectors                  254.0  235.9374              102.3   
ARF without detectors               492.7  120.9500              641.4   

                                 
                            std  
model                            
ARF with detectors     118.0754  
ARF without detectors  223.1493

## 5. Hyperparameter sensitivity

The paper uses River's defaults (10 trees, grace period 50, information gain). Here the
online ARF is rerun with every combination below. Because this is evaluated on the test
stream, it is a sensitivity check, not a way to choose the parameters.

In [7]:
GRID = list(itertools.product([5, 10, 20], [25, 50, 100], ["info_gain", "gini"]))

rows = []
for n_models, grace_period, criterion in GRID:
    for seed in range(GRID_SEEDS):
        X_stream, y_stream, starts = streams[seed]
        model = forest.ARFClassifier(n_models=n_models, max_features=2, seed=seed,
                                     grace_period=grace_period, split_criterion=criterion)
        model = train_river_model(model, X_train, y_train)
        preds, _ = run_stream(model, X_stream, y_stream)
        rows.append({"trees": n_models, "grace period": grace_period,
                     "split criterion": criterion, "seed": seed,
                     "errors": int((preds != y_stream).sum())})
    print(n_models, grace_period, criterion, "done")

grid_results = pd.DataFrame(rows)
grid_results.to_csv("results_hyperparameter_grid.csv", index=False)

grid_table = (grid_results.groupby(["trees", "grace period", "split criterion"])["errors"]
              .agg(["mean", "std"]).round(1).sort_values("mean"))
grid_table["default"] = [key == (10, 50, "info_gain") for key in grid_table.index]
grid_table

5 25 info_gain done
5 25 gini done
5 50 info_gain done
5 50 gini done
5 100 info_gain done
5 100 gini done
10 25 info_gain done
10 25 gini done
10 50 info_gain done
10 50 gini done
10 100 info_gain done
10 100 gini done
20 25 info_gain done
20 25 gini done
20 50 info_gain done
20 50 gini done
20 100 info_gain done
20 100 gini done


mean   std  default
trees grace period split criterion                      
20    100          gini              93.2  12.4    False
                   info_gain         97.4   8.5    False
      50           gini              97.8   8.7    False
10    50           gini              99.2  13.2    False
      100          gini              99.8  12.9    False
20    25           gini             104.2   9.7    False
10    100          info_gain        105.4   5.9    False
      50           info_gain        106.0  15.3     True
20    50           info_gain        106.0   3.7    False
10    25           info_gain        108.2  12.5    False
20    25           info_gain        108.2   5.1    False
10    25           gini             109.0  13.7    False
5     100          gini             110.6   6.3    False
      25           gini             116.8  13.7    False
      50           info_gain        117.8  25.0    False
      100          info_gain        120.6  15.7    False
      50           gini             125.2  17.8    False
      25           info_gain        127.6  14.9    False

In [8]:
learner_results.groupby("model")[["recovery after EDFA", "recovery after NLI"]].count()

,recovery after EDFA,recovery after NLI
model,,
ARF,10,10
Gaussian naive Bayes,10,10
Hoeffding adaptive tree,10,10
Hoeffding tree,10,10
Incremental MLP,10,10
Leveraging bagging,10,10
Streaming random patches,10,9
kNN (window 500),10,10
